In [206]:
import pandas as pd
import numpy as np
import re

In [207]:
data_path = "../data/WWW Bottom Mold list- All brands - 2026 (2).xlsx"

raw = pd.read_excel(data_path)

In [208]:
def clean_df_col_names(df):
    
    cols = {}
    for col in df.columns:
        new_col = str(col).split("\n")[0].strip()
        cols[col] = new_col.strip().lower().replace(' ', '_').replace('-', '_')
    df.rename(columns=cols, inplace=True)
    return df

def replace_whitespace(df):
    return df.replace(
    to_replace=[
        r"^\s*$",     # empty or whitespace-only
        r"^NA$",      # NA
        r"^N/A$",     # N/A
        r"^\xa0$",    # non-breaking space
    ],
    value=np.nan,
    regex=True
)

def normalize_season(value):

    if pd.isna(value):
        return np.nan

    value = str(value).strip().upper()

    # -----------------------------------
    # Normalize season prefixes
    # -----------------------------------
    replacements = {
        "SPR": "S",
        "SS": "S",
        "AW": "F",
        "FW": "F"
    }

    for old, new in replacements.items():
        if value.startswith(old):
            value = value.replace(old, new, 1)

    # -----------------------------------
    # Validate final format
    # -----------------------------------
    pattern = r'^[SF]\d{2}$'

    if re.match(pattern, value):
        return value

    return np.nan

import numpy as np
import pandas as pd


def normalize_location(value):

    if pd.isna(value):
        return np.nan

    value = str(value).strip().title()

    # -----------------------------------
    # Normalize aliases
    # -----------------------------------
    replacements = {
        "North Vietnam": "Vietnam",
        "South Vietnam": "Vietnam",
        "Viet Nam": "Vietnam",
        "Indonesia ": "Indonesia"
    }

    return replacements.get(value, value)

In [209]:
df =raw.copy()

df = clean_df_col_names(df) # Clean column names
df = replace_whitespace(df) # replace whitespaces

# Define data types for each column
dtype_set ={
    'brand': str, 
    'part_name': str, 
    'mold_code': str, 
    'compound': str, 
    'vendor_name': str,
    'location': str, 
    'mold_shop': str, 
    'season': str, 
    'a_mold_cost': float, 
    'last_demand_season': str,
    'mold_ownership': str, 
    'mold_condition': str, 
    'daily_output': float, 
    'total_mold_qty': float,
    '1': float, '1.5': float, '2': float, '2.5': float, '3': float, '3.5': float, '4': float, '4.5': float, '5': float, '5.5': float, '6': float, '6.5': float,
    '7': float, '7.5': float, '8': float, '8.5': float, '9': float, '9.5': float, '10': float, '10.5': float, '11': float, '11.5': float, '12': float,
    '17.5': float, '18': float, 
    'comments': str, 'remark': str, 'actual_output': str
}
for col, dtype in dtype_set.items():
    if col in df.columns and dtype == str:
        df[col] = df[col].astype(dtype)
    elif col in df.columns and dtype == float:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    else:
        print(f"Warning: Column '{col}' not found in DataFrame.")


# keep only valid season values in 'last_demand_season' & 'season' column
df['season'] = df['season'].apply(normalize_season)
df['last_demand_season'] = df['last_demand_season'].apply(normalize_season)

# split brand
df['brand'] = df['brand'].apply(lambda x: x.split("/"))

# normalize vendor_name
df['vendor_name'] = df['vendor_name'].str.strip().str.upper()

#normalize location
df['location'] = df['location'].apply(normalize_location)

In [219]:
df = df[(df['part_name'] =="Outsole") | (df['part_name'] =="Midsole")]

In [220]:
import pandas as pd
import numpy as np
from collections import defaultdict

size_cols = [str(x).replace(".0", "") for x in np.arange(1, 18.5, 0.5)]

result = {}

for _, row in df.iterrows():

    mold_code = row["mold_code"]
    brand = row["brand"]
    part_name = row["part_name"]
    component_name = row["component_name"]

    # ==================================================
    # Create mold_code level
    # ==================================================
    if mold_code not in result:
        result[mold_code] = {
            "brands": brand,
        }

    mold_entry = result[mold_code]

    # ==================================================
    # Create part level
    # ==================================================
    if part_name not in mold_entry:
        mold_entry[part_name] = {}

    part_entry = mold_entry[part_name]

    # ==================================================
    # Create component level
    # ==================================================
    if component_name not in part_entry:
        part_entry[component_name] = {
            "compound": row["compound"],
            "mold_list": []
        }

    component_entry = part_entry[component_name]

    # ==================================================
    # Build size_qty
    # ==================================================
    size_qty = {}

    for size in size_cols:
        val = row.get(size)

        if pd.notna(val):
            try:
                size_qty[size] = int(float(val))
            except:
                size_qty[size] = val

    # ==================================================
    # Mold detail object
    # ==================================================
    mold_detail = {
        "vendor": row["vendor_name"],
        "location": row["location"],
        "mold_shop": row["mold_shop"],
        "init_season": row["season"],

        "daily_output": row["daily_output"],
        "total_mold_qty": row["total_mold_qty"],
        "actual_output": row["actual_output"],

        "mold_cost": row["a_mold_cost"],
        "last_demand_season": row["last_demand_season"],
        "mold_ownership": row["mold_ownership"],
        "mold_condition": row["mold_condition"],
        "mold_condition_note": row["mold_condition_note"],

        "comments": row["comments"],
        "remark": row["remark"],

        "size_qty": size_qty
    }

    component_entry["mold_list"].append(mold_detail)

In [221]:
import json

with open("../data/mold_data.json", "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False, default=str)